# POC: Extração de Documentos com IA Generativa
### Desafio Técnico · Tech for Humans · Marcello Siqueira

Esta POC demonstra a Abordagem C (Pipeline Híbrida Modular) recomendada no Dossiê de Pesquisa,
implementada com **Google Gemini 2.0 Flash** via chamadas HTTP diretas à API REST.

**Casos de uso:**
- Caso 1: CNH, extração de campos predeterminados via JSON schema
- Caso 2: Paper Claude 3 (42 páginas), extração de conteúdo preservando layout
- Caso 3: Fatura de Energia CELPE, extração estruturada completa com histograma


## 1. Setup

In [25]:
pip install pdf2image Pillow requests


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import json
import base64
from pathlib import Path
from io import BytesIO

import requests
from pdf2image import convert_from_path
from PIL import Image

# Cole aqui sua chave do Google AI Studio (aistudio.google.com)
GOOGLE_API_KEY = "SUA_CHAVE_AQUI"

MODEL    = "gemini-2.5-flash"
BASE_URL = (
    f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}"
    f":generateContent?key={GOOGLE_API_KEY}"
)

print(f"Modelo: {MODEL}")
print("Usando API REST direta (sem gRPC)")


Modelo: gemini-2.5-flash
Usando API REST direta (sem gRPC)


## 2. Funções Auxiliares

In [27]:
def imagem_para_b64(img: Image.Image, max_size: int = 1568) -> str:
    """
    Converte um objeto PIL Image para base64 JPEG.
    Redimensiona se a maior dimensão ultrapassar max_size pixels.
    Isso reduz o consumo de tokens sem perda de legibilidade.
    """
    w, h = img.size
    if max(w, h) > max_size:
        scale = max_size / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

    buf = BytesIO()
    img.save(buf, format="JPEG", quality=88)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def carregar_imagem(path: str, max_size: int = 1568) -> str:
    """Carrega uma imagem do disco e retorna como base64 JPEG."""
    img = Image.open(path).convert("RGB")
    return imagem_para_b64(img, max_size)


def chamar_gemini(partes: list) -> str:
    payload = {"contents": [{"parts": partes}]}

    resp = requests.post(BASE_URL, json=payload, timeout=120)

    if resp.status_code != 200:
        raise RuntimeError(
            f"Erro na API: {resp.status_code}\n{resp.text[:500]}"
        )

    dados = resp.json()
    candidato = dados["candidates"][0]

    # Gemini pode bloquear por filtros de segurança sem retornar content
    if "content" not in candidato:
        motivo = candidato.get("finishReason", "desconhecido")
        print(f"  Aviso: resposta bloqueada pelo modelo (motivo: {motivo}). Retornando vazio.")
        return ""

    return candidato["content"]["parts"][0]["text"]


def parte_imagem(b64: str) -> dict:
    """Formata uma imagem base64 no formato de parte da API Gemini."""
    return {"inline_data": {"mime_type": "image/jpeg", "data": b64}}


def parte_texto(texto: str) -> dict:
    """Formata um texto no formato de parte da API Gemini."""
    return {"text": texto}


def extrair_com_schema(b64_imagens: list[str], schema: dict, prompt: str) -> dict:
    """
    Envia imagens e um schema JSON para o Gemini e retorna um dict estruturado.

    O schema é incluído no prompt para que o modelo saiba exatamente qual
    estrutura JSON deve retornar. O resultado é parseado com json.loads().

    Args:
        b64_imagens: lista de strings base64 das imagens
        schema:      JSON schema que define a estrutura esperada
        prompt:      instrução principal de extração

    Returns:
        dict com os campos extraídos
    """
    schema_str = json.dumps(schema, ensure_ascii=False, indent=2)

    instrucao = (
        f"{prompt}\n\n"
        "Retorne APENAS um objeto JSON válido seguindo exatamente este schema:\n"
        f"{schema_str}\n\n"
        "Não inclua explicações, markdown, blocos de código ou texto fora do JSON."
    )

    partes = [parte_imagem(b64) for b64 in b64_imagens]
    partes.append(parte_texto(instrucao))

    resposta = chamar_gemini(partes)

    # Remove blocos ```json ... ``` caso o modelo os inclua
    texto = resposta.strip()
    if texto.startswith("```"):
        linhas = texto.split("\n")
        linhas = [l for l in linhas if not l.strip().startswith("```")]
        texto  = "\n".join(linhas).strip()

    return json.loads(texto)


def extrair_texto_lote(paginas_b64: list[str], instrucao: str, inicio: int = 0) -> str:
    """
    Envia um lote de páginas como imagens base64 e recebe Markdown estruturado.
    Usado no Caso 2 para processar o PDF em partes.

    Args:
        paginas_b64: lista de strings base64, uma por página
        instrucao:   prompt de extração
        inicio:      número da primeira página do lote

    Returns:
        string com o conteúdo extraído em Markdown
    """
    partes = []
    for i, b64 in enumerate(paginas_b64):
        partes.append(parte_texto(f"Página {inicio + i + 1}"))
        partes.append(parte_imagem(b64))
    partes.append(parte_texto(instrucao))

    return chamar_gemini(partes)


print("Funções auxiliares carregadas.")


Funções auxiliares carregadas.


## Caso 1: Documento Estruturado (CNH)

**Objetivo:** extrair os campos predeterminados da CNH em JSON estruturado.

Campos obrigatórios: Nome, CPF, Data de Nascimento, Data de Emissão, Filiação Pai, Filiação Mãe.


In [28]:
SCHEMA_CNH = {
    "type": "object",
    "properties": {
        "nome":            {"type": "string",  "description": "Nome completo do portador"},
        "cpf":             {"type": "string",  "description": "CPF no formato XXX.XXX.XXX-XX"},
        "data_nascimento": {"type": "string",  "description": "Data de nascimento DD/MM/AAAA"},
        "data_emissao":    {"type": "string",  "description": "Data de emissão DD/MM/AAAA"},
        "filiacao_pai":    {"type": "string",  "description": "Nome completo do pai"},
        "filiacao_mae":    {"type": "string",  "description": "Nome completo da mãe"},
        "numero_registro": {"type": "string",  "description": "Número do registro"},
        "data_validade":   {"type": "string",  "description": "Data de validade DD/MM/AAAA"},
        "categoria":       {"type": "string",  "description": "Categoria da CNH (A, B, AB, etc.)"},
        "local_emissao":   {"type": "string",  "description": "Cidade e UF de emissão"}
    },
    "required": ["nome", "cpf", "data_nascimento", "data_emissao", "filiacao_pai", "filiacao_mae"]
}

PROMPT_CNH = (
    "Analise a imagem desta CNH brasileira com atenção máxima. "
    "Extraia todos os campos visíveis exatamente como aparecem no documento. "
    "Mantenha a formatação original de CPF e datas. "
    "Se algum campo obrigatório não estiver legível, escreva: nao legivel."
)

CNH_PATH = "Documento 1.jpeg"

print(f"Carregando {CNH_PATH}...")
b64_cnh = carregar_imagem(CNH_PATH)

print("Enviando para o Gemini...")
resultado_cnh = extrair_com_schema(
    b64_imagens=[b64_cnh],
    schema=SCHEMA_CNH,
    prompt=PROMPT_CNH
)

print(json.dumps(resultado_cnh, ensure_ascii=False, indent=2))


Carregando Documento 1.jpeg...
Enviando para o Gemini...
{
  "nome": "LINCE DA SILVA",
  "cpf": "891.340.611-75",
  "data_nascimento": "02/05/2017",
  "data_emissao": "22/10/2013",
  "filiacao_pai": "Pai José da Silva",
  "filiacao_mae": "Mãe Maria da Silva",
  "numero_registro": "00123456789",
  "data_validade": "02/05/2018",
  "categoria": "B",
  "local_emissao": "BRASÍLIA - DISTRITO FEDERAL, DF"
}


In [29]:
campos_obrigatorios = [
    "nome", "cpf", "data_nascimento",
    "data_emissao", "filiacao_pai", "filiacao_mae"
]

print("Validação dos campos obrigatórios:")
todos_ok = True
for campo in campos_obrigatorios:
    valor  = resultado_cnh.get(campo, "AUSENTE")
    status = "OK" if valor and valor not in ("AUSENTE", "nao legivel") else "FALTANDO"
    if status == "FALTANDO":
        todos_ok = False
    print(f"  [{status}] {campo}: {valor}")

print()
if todos_ok:
    print("Todos os campos obrigatórios foram extraídos com sucesso.")
else:
    print("Atenção: há campos ausentes ou ilegíveis.")


Validação dos campos obrigatórios:
  [OK] nome: LINCE DA SILVA
  [OK] cpf: 891.340.611-75
  [OK] data_nascimento: 02/05/2017
  [OK] data_emissao: 22/10/2013
  [OK] filiacao_pai: Pai José da Silva
  [OK] filiacao_mae: Mãe Maria da Silva

Todos os campos obrigatórios foram extraídos com sucesso.


## Caso 2: Documento Extenso (Paper Claude 3, 42 páginas)

**Objetivo:** extrair o conteúdo completo para texto, mantendo hierarquia de seções,
tabelas em Markdown e interpretação de gráficos e figuras.

O PDF é convertido em imagens pelo `pdf2image` e processado em lotes de 6 páginas por chamada.


In [30]:
PDF_PATH   = "Documento 3.pdf"
DPI        = 120
BATCH_SIZE = 6

print(f"Convertendo o PDF em imagens com DPI={DPI}...")
paginas_pil = convert_from_path(PDF_PATH, dpi=DPI)
print(f"{len(paginas_pil)} páginas convertidas.")

# Pré-converte todas as páginas para base64 para evitar reconversão nos lotes
print("Codificando páginas em base64...")
paginas_b64 = [imagem_para_b64(pg) for pg in paginas_pil]
print("Pronto.")


Convertendo o PDF em imagens com DPI=120...
42 páginas convertidas.
Codificando páginas em base64...
Pronto.


In [31]:
INSTRUCAO_PAPER = """Você está recebendo páginas de um artigo técnico sobre o modelo Claude 3 da Anthropic.
Para cada página recebida:

1. Extraia todo o texto preservando a hierarquia com Markdown:
   use # para título principal, ## para seções e ### para subseções.
2. Reproduza todas as tabelas em formato Markdown com pipes (| coluna | coluna |).
3. Para gráficos, diagramas e figuras, insira um parágrafo iniciando com
   [FIGURA: descrição] explicando o conteúdo, eixos, valores e tendências visíveis.
4. Mantenha a ordem de leitura original do documento.
5. Separe cada página com a linha: --- Fim da Página N ---

Não omita nenhum conteúdo. Seja fiel ao original.
"""

conteudo_completo = []
total_lotes = (len(paginas_b64) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Processando {total_lotes} lotes de até {BATCH_SIZE} páginas cada.")

for i in range(0, len(paginas_b64), BATCH_SIZE):
    lote       = paginas_b64[i : i + BATCH_SIZE]
    num_lote   = i // BATCH_SIZE + 1
    pag_inicio = i + 1
    pag_fim    = min(i + BATCH_SIZE, len(paginas_b64))

    print(f"Lote {num_lote}/{total_lotes}: páginas {pag_inicio} a {pag_fim}...", end=" ")

    texto = extrair_texto_lote(
        paginas_b64=lote,
        instrucao=INSTRUCAO_PAPER,
        inicio=i
    )
    conteudo_completo.append(texto)
    print("concluído.")

texto_final = "\n\n".join(conteudo_completo)
print(f"\nExtração concluída. Total de caracteres: {len(texto_final):,}")


Processando 7 lotes de até 6 páginas cada.
Lote 1/7: páginas 1 a 6... concluído.
Lote 2/7: páginas 7 a 12... concluído.
Lote 3/7: páginas 13 a 18... concluído.
Lote 4/7: páginas 19 a 24... concluído.
Lote 5/7: páginas 25 a 30... concluído.
Lote 6/7: páginas 31 a 36... concluído.
Lote 7/7: páginas 37 a 42... concluído.

Extração concluída. Total de caracteres: 211,092


In [32]:
print(texto_final[:3000])
if len(texto_final) > 3000:
    print(f"\n... {len(texto_final) - 3000:,} caracteres adicionais não exibidos ...")


# The Claude 3 Model Family: Opus, Sonnet, Haiku

### Anthropic

### Abstract
We introduce Claude 3, a new family of large multimodal models – Claude 3 Opus, our most capable offering, **Claude 3 Sonnet**, which provides a combination of skills and speed, and **Claude 3 Haiku**, our fastest and least expensive model. All new models have vision capabilities that enable them to process and analyze image data. The Claude 3 family demonstrates strong performance across benchmark evaluation and sets a new standard on measures of reasoning, math, and coding. Claude 3 Opus achieves state-of-the-art results on evaluation for GPOA [1], MMMLU [2], MMMU [3], and many more. Claude 3 Haiku performs as well or better than Claude 2 [4] on most pure-text tasks, while Sonnet and Opus significantly outperform it. Additionally, these models exhibit improved fluency in non-English languages, making them more versatile for a global audience. In this report, we provide an in-depth analysis of our evaluation

In [33]:
output_md = "caso2_paper_claude3_extraido.md"
Path(output_md).write_text(texto_final, encoding="utf-8")
print(f"Arquivo salvo: {output_md}")
print(f"Tamanho: {Path(output_md).stat().st_size / 1024:.1f} KB")


Arquivo salvo: caso2_paper_claude3_extraido.md
Tamanho: 207.6 KB


## Caso 3: Layout Complexo (Fatura de Energia CELPE)

**Objetivo:** extrair todo o conteúdo da fatura mantendo a organização hierárquica:
dados do cliente, medição, tributos, composição do consumo e histórico mensal.


In [34]:
SCHEMA_FATURA = {
    "type": "object",
    "properties": {
        "distribuidora": {"type": "string"},
        "dados_cliente": {
            "type": "object",
            "properties": {
                "nome":              {"type": "string"},
                "cpf_cnpj":          {"type": "string"},
                "endereco":          {"type": "string"},
                "numero_instalacao": {"type": "string"},
                "numero_cliente":    {"type": "string"},
                "conta_contrato":    {"type": "string"},
                "classificacao":     {"type": "string"}
            }
        },
        "dados_fatura": {
            "type": "object",
            "properties": {
                "mes_referencia":       {"type": "string"},
                "data_emissao":         {"type": "string"},
                "data_vencimento":      {"type": "string"},
                "data_proxima_leitura": {"type": "string"},
                "total_a_pagar_rs":     {"type": "number"},
                "numero_nota_fiscal":   {"type": "string"}
            }
        },
        "medicao": {
            "type": "object",
            "properties": {
                "numero_medidor":         {"type": "string"},
                "leitura_anterior":       {"type": "number"},
                "data_leitura_anterior":  {"type": "string"},
                "leitura_atual":          {"type": "number"},
                "data_leitura_atual":     {"type": "string"},
                "numero_dias":            {"type": "integer"},
                "consumo_kwh":            {"type": "number"},
                "constante":              {"type": "number"}
            }
        },
        "tributos": {
            "type": "object",
            "properties": {
                "icms_base_calculo": {"type": "number"},
                "icms_percentual":   {"type": "number"},
                "icms_valor":        {"type": "number"},
                "pis_percentual":    {"type": "number"},
                "pis_valor":         {"type": "number"},
                "cofins_percentual": {"type": "number"},
                "cofins_valor":      {"type": "number"}
            }
        },
        "composicao_consumo": {
            "type": "object",
            "properties": {
                "geracao_energia_rs":     {"type": "number"},
                "geracao_energia_pct":    {"type": "number"},
                "transmissao_rs":         {"type": "number"},
                "transmissao_pct":        {"type": "number"},
                "distribuicao_rs":        {"type": "number"},
                "distribuicao_pct":       {"type": "number"},
                "encargos_setoriais_rs":  {"type": "number"},
                "encargos_setoriais_pct": {"type": "number"},
                "tributos_rs":            {"type": "number"},
                "tributos_pct":           {"type": "number"},
                "total_rs":               {"type": "number"}
            }
        },
        "historico_consumo": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "mes_ano":     {"type": "string"},
                    "consumo_kwh": {"type": "number"}
                }
            }
        },
        "niveis_tensao": {
            "type": "object",
            "properties": {
                "tensao_nominal_v": {"type": "number"},
                "limite_minimo_v":  {"type": "number"},
                "limite_maximo_v":  {"type": "number"}
            }
        },
        "qualidade_fornecimento": {
            "type": "object",
            "properties": {
                "dic_apurado":  {"type": "number"},
                "fic_apurado":  {"type": "number"},
                "dmic_apurado": {"type": "number"}
            }
        }
    },
    "required": ["distribuidora", "dados_cliente", "dados_fatura"]
}

PROMPT_FATURA = (
    "Analise esta fatura de energia elétrica com atenção máxima. "
    "Extraia todos os campos visíveis: dados do cliente, cobrança, medição, "
    "tributos (ICMS, PIS, COFINS), composição percentual do consumo, "
    "histórico mensal do gráfico de barras (todos os meses da esquerda para a direita), "
    "indicadores de qualidade (DIC, FIC, DMIC) e níveis de tensão. "
    "Para valores monetários use número decimal (ex: 63.72 e não R$ 63,72)."
)

FATURA_PATH = "Documento 2.jpg"

print(f"Carregando {FATURA_PATH}...")
b64_fatura = carregar_imagem(FATURA_PATH, max_size=2048)

print("Enviando para o Gemini...")
resultado_fatura = extrair_com_schema(
    b64_imagens=[b64_fatura],
    schema=SCHEMA_FATURA,
    prompt=PROMPT_FATURA
)

print(json.dumps(resultado_fatura, ensure_ascii=False, indent=2))


Carregando Documento 2.jpg...
Enviando para o Gemini...
{
  "distribuidora": "Celpe - Companhia Energética de Pernambuco",
  "dados_cliente": {
    "nome": "MARIA JOSÉ DA SILVA",
    "cpf_cnpj": "123.456.789-10",
    "endereco": "RUA JOÃO FERNANDES VIEIRA 175 BOA VISTA",
    "numero_instalacao": "1234567890",
    "numero_cliente": "1234567890",
    "conta_contrato": "1234567890",
    "classificacao": "BI RESIDENCIAL RESIDENCIAL MONOFÁSICO"
  },
  "dados_fatura": {
    "mes_referencia": "06/2014",
    "data_emissao": "26/06/2014",
    "data_vencimento": "10/07/2014",
    "data_proxima_leitura": "25/07/2014",
    "total_a_pagar_rs": 63.72,
    "numero_nota_fiscal": "00000251"
  },
  "medicao": {
    "numero_medidor": "1234567890",
    "leitura_anterior": 24630.0,
    "data_leitura_anterior": "27/05/2014",
    "leitura_atual": 24790.0,
    "data_leitura_atual": "26/06/2014",
    "numero_dias": 31,
    "consumo_kwh": 160.0,
    "constante": 1.0
  },
  "tributos": {
    "icms_base_calculo":

In [35]:
historico = resultado_fatura.get("historico_consumo", [])

if historico:
    print(f"Histórico de consumo ({len(historico)} meses)\n")
    maior = max(item.get("consumo_kwh", 1) for item in historico)
    for item in historico:
        mes   = item.get("mes_ano", "?")
        kwh   = item.get("consumo_kwh", 0)
        barra = "█" * max(1, int(30 * kwh / maior))
        print(f"  {mes:8s} {str(kwh):6s} kWh  {barra}")
else:
    print("Histórico de consumo não encontrado no resultado.")


Histórico de consumo (13 meses)

  JUL 14   156    kWh  █████████████████████████
  JUN 14   160    kWh  ██████████████████████████
  MAI 14   155    kWh  █████████████████████████
  ABR 14   158    kWh  ██████████████████████████
  MAR 14   152    kWh  █████████████████████████
  FEV 14   157    kWh  █████████████████████████
  JAN 14   151    kWh  ████████████████████████
  DEZ 13   155    kWh  █████████████████████████
  NOV 13   166    kWh  ███████████████████████████
  OUT 13   174    kWh  ████████████████████████████
  SET 13   182    kWh  ██████████████████████████████
  AGO 13   165    kWh  ███████████████████████████
  JUL 13   156    kWh  █████████████████████████


## Resumo da POC

| Caso | Documento | Abordagem | Resultado |
|---|---|---|---|
| 1 | CNH | JSON schema no prompt | Campos extraídos em JSON validado |
| 2 | Paper Claude 3 (42 pág.) | Lotes de imagens + Markdown | Texto estruturado com tabelas e figuras |
| 3 | Fatura CELPE | JSON schema hierárquico | 8 seções extraídas incluindo histograma |

**Comparação com o pipeline atual:**

| Dimensão | Pipeline Atual (2 LLMs) | Esta POC (Gemini 2.0 Flash) |
|---|---|---|
| Chamadas de API por documento | 2 | 1 (Casos 1 e 3) |
| Output JSON garantido | Não | Sim (schema no prompt) |
| Interpretação de gráficos | Limitada | Nativa |
| Custo por 1.000 páginas | US$ 18 a 25 | Gratuito no tier free |


In [36]:
print("POC concluída.")
print()
print(f"Caso 1 (CNH):    {len(resultado_cnh)} campos extraídos")
print(f"Caso 2 (Paper):  {len(paginas_pil)} páginas, {len(texto_final):,} caracteres")
print(f"Caso 3 (Fatura): {len(resultado_fatura)} seções, "
      f"{len(resultado_fatura.get('historico_consumo', []))} meses no histórico")


POC concluída.

Caso 1 (CNH):    10 campos extraídos
Caso 2 (Paper):  42 páginas, 211,092 caracteres
Caso 3 (Fatura): 9 seções, 13 meses no histórico
